# 0. Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import warnings

# Filter out the specific UserWarnings
warnings.filterwarnings("ignore", category=UserWarning, message="A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy")
warnings.filterwarnings("ignore", category=UserWarning, message="unable to load libtensorflow_io_plugins.so")
warnings.filterwarnings("ignore", category=UserWarning, message="file system plugins are not loaded")

In [3]:
# Hugging Face library
from transformers import AutoTokenizer, TFAutoModel

c:\Users\feder\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Hugging Face library
from datasets import Dataset, DatasetDict

In [5]:
# Accuracy metrics from Scikit-Learn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report

In [6]:
import tensorflow as tf

from tensorflow.keras.optimizers import AdamW

from tensorflow.keras.losses import SparseCategoricalCrossentropy

from tensorflow.keras.layers import Input, Dense, GlobalMaxPooling1D, LSTM, Dropout

from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler

# 1. Load Datasets

In [7]:
# Create a function to import the data from csv format
def load_data(file_path):
    return pd.read_csv(file_path, header=None, delimiter='\t', names=['sentiment', 'text'])


train_path = r"C:\Users\feder\Desktop\Project\Bertoia (Master Thesis)\Data\train_bal_vdg_27_11.tsv"
test_path = r"C:\Users\feder\Desktop\Project\Bertoia (Master Thesis)\Data\test_bal_vdg_27_11.tsv"
val_path = r"C:\Users\feder\Desktop\Project\Bertoia (Master Thesis)\Data\valid_bal_vdg_27_11.tsv"

df_train = load_data(train_path)
df_test = load_data(test_path)
df_val = load_data(val_path)

In [8]:
# Since I'm gonna use the sparse categorical cross entropy loss, I map the labels to integers
encoded_dict = {'NEG':0, 'NEU':1, 'POS':2}

df_train['label'] = df_train['sentiment'].apply(lambda x: encoded_dict[x])
df_test['label'] = df_test['sentiment'].apply(lambda x: encoded_dict[x])
df_val['label'] = df_val['sentiment'].apply(lambda x: encoded_dict[x])


In [9]:
# To get an idea of the data
pd.set_option('display.max_colwidth', 150)
df_train.head()

,sentiment,text,label
0,NEU,"La lotta contro il bodyshaming è una cosa, promuovere l’obesità è un’altra",1
1,POS,"La marginalità è un luogo radicale di possibilità, uno spazio di #resistenza. Ne parla @Racheleborghi in questo podcast pubblicato da #TRANSfemmIN...",2
2,NEU,"@ilgiomba @GiovannaSerra3 @voilaloves @ArthurMeurs @LuniVale @antrichelieu @diegodemme4 @fabyo255 Io spero solo,che in cuor suo,Giovanna abbia ca...",1
3,POS,Seppellire l'odio sotto una montagna di amore #ProudBoys 🧡💜💙💚♥️,2
4,NEU,#iorestoacasama non dimentico. E SE IO LOTTO DA PARTIGIANA Raccolta delle biografie delle partigiane a cura di @NonUnaDiMenoMI https://t.co/KUlwqE...,1


In [10]:
# Both these functions can have as input a single label/id or a list of them

def label2id(label):
    if isinstance(label, list):
        return [encoded_dict[label] for label in label]
    else:
        return encoded_dict[label]

def id2label(id):
    encoded_dict_inv = {v: k for k, v in encoded_dict.items()}
    
    if isinstance(id, list):
        return [encoded_dict_inv[i] for i in id]
    else:
        return encoded_dict_inv[id]

In [11]:
# I'm combining the pandas dataframe to the dataset dictionary of Hugging Face

train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)
val_dataset = Dataset.from_pandas(df_val)

# Create the DatasetDict
dataset = DatasetDict({'train': train_dataset, 'test': test_dataset, 'validation': val_dataset})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sentiment', 'text', 'label'],
        num_rows: 717
    })
    test: Dataset({
        features: ['sentiment', 'text', 'label'],
        num_rows: 216
    })
    validation: Dataset({
        features: ['sentiment', 'text', 'label'],
        num_rows: 92
    })
})


In [12]:
# Removing duplicates

# Initialize a dictionary to store updated datasets
updated_datasets = {}

# Check for and remove duplicates in each split
for split in dataset.keys():
    split_data = dataset[split]
    
    # Access the 'text' column within the list
    text_column = split_data['text']
    
    # Initialize a set to track unique texts
    unique_texts = set()
    
    # Initialize lists to store the filtered data
    filtered_text = []
    
    # Iterate through the 'text' column and filter duplicates
    for text in text_column:
        if text not in unique_texts:
            unique_texts.add(text)
            filtered_text.append(text)
    
    # Create a new Dataset object with the filtered data
    updated_datasets[split] = split_data.select(list(range(len(filtered_text))))
    
    # Print the number of removed duplicates
    duplicate_count = len(text_column) - len(filtered_text)
    print(f"Duplicates removed in {split} split: {duplicate_count}\n")

# Update the dataset dictionary with the filtered datasets
dataset.update(updated_datasets)

# Print the updated dataset information
for split in dataset.keys():
    split_data = dataset[split]
    print(f"{split}: {len(split_data['text'])} rows\n")

print(dataset)

Duplicates removed in train split: 1

Duplicates removed in test split: 0

Duplicates removed in validation split: 1

train: 716 rows

test: 216 rows

validation: 91 rows

DatasetDict({
    train: Dataset({
        features: ['sentiment', 'text', 'label'],
        num_rows: 716
    })
    test: Dataset({
        features: ['sentiment', 'text', 'label'],
        num_rows: 216
    })
    validation: Dataset({
        features: ['sentiment', 'text', 'label'],
        num_rows: 91
    })
})


# 2. Preprocess data

In [13]:
bert = TFAutoModel.from_pretrained('dbmdz/bert-base-italian-xxl-cased')
tokenizer = AutoTokenizer.from_pretrained('dbmdz/bert-base-italian-xxl-cased')

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [14]:
max_length = 256

def tokenize_text(dataset):
    return tokenizer(
        text=dataset['text'],
        add_special_tokens=True,
        return_token_type_ids=False,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='tf',
        verbose=True
    )

In [15]:
encoded_dataset = dataset.map(tokenize_text)

In [16]:
print(encoded_dataset)

DatasetDict({
    train: Dataset({
        features: ['sentiment', 'text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 716
    })
    test: Dataset({
        features: ['sentiment', 'text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 216
    })
    validation: Dataset({
        features: ['sentiment', 'text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 91
    })
})


In [17]:
encoded_dataset = encoded_dataset.remove_columns(['sentiment','text'])

encoded_dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 716
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 216
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 91
    })
})

In [18]:
print(encoded_dataset['train']['input_ids'][0])

# Note that is a list into a list

[[102, 309, 3188, 593, 162, 1380, 2496, 23669, 11348, 198, 224, 510, 1307, 4723, 181, 5817, 10706, 437, 232, 198, 141, 5817, 5241, 103, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]


In [19]:
def preprocess_data(encoded_dataset, data_type):
    input_ids = np.array(encoded_dataset[data_type]['input_ids'])
    input_ids = np.squeeze(input_ids, axis=1)

    attention_mask = np.array(encoded_dataset[data_type]['attention_mask'])
    attention_mask = np.squeeze(attention_mask, axis=1)

    label = np.array(encoded_dataset[data_type]['label'])

    return input_ids, attention_mask, label

def main_processing(encoded_dataset):
    input_ids_train, attention_mask_train, label_train = preprocess_data(encoded_dataset, 'train')
    input_ids_val, attention_mask_val, label_val = preprocess_data(encoded_dataset, 'validation')
    input_ids_test, attention_mask_test, label_test = preprocess_data(encoded_dataset, 'test')

    return input_ids_train, attention_mask_train, label_train, input_ids_val, attention_mask_val, label_val, input_ids_test, attention_mask_test, label_test

# Usage
input_ids_train, attention_mask_train, label_train, input_ids_val, attention_mask_val, label_val, input_ids_test, attention_mask_test, label_test = main_processing(encoded_dataset)


In [20]:
print(input_ids_train[0])

[  102   309  3188   593   162  1380  2496 23669 11348   198   224   510
  1307  4723   181  5817 10706   437   232   198   141  5817  5241   103
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0   

# 3. Defining the model

In [21]:
input_ids = Input(shape=(max_length,), dtype=tf.int32, name="input_ids")
input_mask = Input(shape=(max_length,), dtype=tf.int32, name="attention_mask")

embeddings = bert.bert(input_ids, attention_mask = input_mask, output_hidden_states=True)[2]

In [22]:
embeddings

(<KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dtype=float32 (created by layer 'bert')>,
 <KerasTensor: shape=(None, 256, 768) dt

## 3.1 Pooler Output [CLS]

In [ ]:
out = Dense(128, activation='relu',name='Dense_relu')(embeddings)
out = Dropout(0.1)(out)
y = Dense(3, activation='softmax',name="Dense_softmax")(out)

model = tf.keras.Model(inputs=[input_ids, input_mask], outputs=y)
model.layers[2].trainable = True


## 3.2 Max Pooling

In [ ]:
out = GlobalMaxPooling1D(name="GlobalMaxPooling1d")(embeddings)
out = Dropout(0.1)(out)
out = Dense(128, activation='relu',name="Dense_relu")(out)


y = Dense(3, activation='softmax',name="Dense_softmax")(out)
    
model = tf.keras.Model(inputs=[input_ids, input_mask], outputs=y)
model.layers[2].trainable = True

## 3.3 LSTM Pooling

In [23]:
class LSTMPooling(tf.keras.layers.Layer):
    def __init__(self, num_layers, hidden_size, hiddendim_lstm):
        super(LSTMPooling, self).__init__()
        self.num_hidden_layers = num_layers
        self.hidden_size = hidden_size
        self.hiddendim_lstm = hiddendim_lstm
        self.lstm = LSTM(self.hiddendim_lstm, return_sequences=True)
        self.dropout = tf.keras.layers.Dropout(0.1)

    def call(self, all_hidden_states):
        hidden_states = tf.stack([all_hidden_states[layer_i][:, 0] for layer_i in range(1, self.num_hidden_layers + 1)], axis=-1)
        hidden_states = tf.reshape(hidden_states, [-1, self.num_hidden_layers, self.hidden_size])
        out = self.lstm(hidden_states)
        out = self.dropout(out[:, -1])
        return out

config = {
    'num_hidden_layers': 12,  
    'hidden_size': 768,
    'hiddendim_lstm': 256
}


In [24]:
pooler = LSTMPooling(config['num_hidden_layers'], config['hidden_size'], config['hiddendim_lstm'])
                                                                                
lstm_pooling = pooler(embeddings)
out = Dense(128, activation='relu',name="Dense_relu")(lstm_pooling)
y = Dense(3, activation='softmax',name="Dense_softmax")(out)
    
model = tf.keras.Model(inputs=[input_ids, input_mask], outputs=y)
model.layers[2].trainable = True

## 4. Fine-Tuning

In [25]:
for i, layer in enumerate(model.layers):
    print(f"Layer {i}: {layer.name}")

Layer 0: input_ids
Layer 1: attention_mask
Layer 2: bert
Layer 3: lstm_pooling
Layer 4: Dense_relu
Layer 5: Dense_softmax


In [26]:
optimizer = AdamW(
    learning_rate=2e-5,
    epsilon=1e-08,
    weight_decay=0.001,
    name="AdamW"
)

In [27]:
def scheduler(epoch,lr):
    if epoch <2:
        return lr
    else:
        return lr*tf.math.exp(-0.1)
    
lr_scheduler = LearningRateScheduler(scheduler)

In [28]:
loss = SparseCategoricalCrossentropy(
    from_logits=False,
    ignore_class=None,
    reduction="auto",
    name="sparse_categorical_crossentropy",
)

In [29]:
early_stop = EarlyStopping(
    monitor="val_loss",
    min_delta=0,
    patience=4,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=True,
    start_from_epoch=0,
)

In [30]:
model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=['sparse_categorical_accuracy']
)

In [31]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, 256)]                0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, 256)]                0         []                            
 )                                                                                                
                                                                                                  
 bert (TFBertMainLayer)      TFBaseModelOutputWithPooli   1106956   ['input_ids[0][0]',           
                             ngAndCrossAttentions(last_   80         'attention_mask[0][0]']      
                             hidden_state=(None, 256, 7                                       

The output shape is shown as "(None, ...)" in the layer summary you provided because the specific batch size dimension is not fixed in the layer summary. In many deep learning frameworks, including TensorFlow, Keras, and others, when you define a model, you typically leave the batch size dimension as "None" in the layer summary. The "None" here indicates that the batch size is not specified at the model definition stage and will be determined dynamically during training or inference based on the input data.

input_ids and attention_mask:

Shape: (None, 256)
Explanation: These input layers are typically used for processing sequences, such as text data. The (None, 256) shape means that the model expects input sequences with a maximum length of 256 tokens, and the batch size can vary (indicated by "None").

bert (TFBertMainLayer):

Output Shape: (None, 256, 768)
Explanation: This is the output shape of the BERT model. It processes input sequences and produces embeddings for each token in the sequence. The first dimension "None" represents the batch size, the second dimension "256" represents the sequence length, and the third dimension "768" represents the size of the hidden representation for each token.

global_max_pooling1d (GlobalMaxPooling1D):

Output Shape: (None, 768)
Explanation: This layer performs global max-pooling over the token embeddings generated by BERT. It takes the maximum value across the sequence length dimension (256) for each of the 768 hidden units, resulting in a fixed-size representation for each input example. The "None" batch dimension remains unspecified.

dense (Dense):

Output Shape: (None, 128)
Explanation: This is a fully connected (dense) layer with 128 output units. It takes the output from the global max-pooling layer and transforms it into a lower-dimensional space. The "None" batch dimension indicates variable batch size.

dropout_37 (Dropout):

Output Shape: (None, 128)
Explanation: Dropout is a regularization technique where a fraction of input units is randomly set to zero during each update, helping to prevent overfitting. The "None" batch dimension remains unspecified.

dense_1 (Dense):

Output Shape: (None, 32)
Explanation: This is another fully connected layer with 32 output units. It further reduces the dimensionality of the data. The "None" batch dimension indicates variable batch size.

dense_2 (Dense):

Output Shape: (None, 3)
Explanation: This is the final dense layer with 3 output units. It produces the final predictions or scores for a classification task with 3 classes. The "None" batch dimension remains unspecified.

# 5. Training

In [ ]:
history = model.fit(
    x = {'input_ids':input_ids_train, 'attention_mask':attention_mask_train},
    y = label_train,
    validation_data = ({'input_ids':input_ids_val, 'attention_mask':attention_mask_val},
                      (label_val)),
    epochs=15,
    batch_size=8,
    callbacks=[early_stop, lr_scheduler]
)

In [ ]:
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

# 6. Metrics

In [ ]:
predicted = model.predict({'input_ids': input_ids_test, 'attention_mask': attention_mask_test})
predicted_labels = np.argmax(predicted, axis=1)

In [ ]:
predicted_labels = predicted_labels.tolist()
predicted_labels = id2label(predicted_labels)
predicted_labels[0:7]

In [ ]:
label_test = label_test.tolist()
label_test = id2label(label_test)
label_test[0:7]

In [ ]:
dataset['test']['text'][0:3]

In [ ]:
print(classification_report(label_test, predicted_labels))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Compute the confusion matrix
conf_matrix = confusion_matrix(label_test, predicted_labels, labels=['NEG', 'NEU', 'POS'])

# Create a heatmap for the confusion matrix
plt.figure(figsize=(8, 6))

sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NEG', 'NEU', 'POS'], 
            yticklabels=['NEG', 'NEU', 'POS'])

plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


## 5.1 Accuracy Score

In [ ]:
accuracy = accuracy_score(label_test, predicted_labels) # (TP+TN)/P+N i.e total number of corrected classified tweet over total number of tweets

print(accuracy)

## 5.2 Preision Score

In [ ]:
precision = precision_score(label_test, predicted_labels,average=None, labels=['NEG','NEU','POS']) # TP/(TP+FP) i.e if predicted a certain class, which is the probability of being really that class?

print(precision)

## 5.3 Recall (sensitivity) Score

In [ ]:
recall = recall_score(label_test, predicted_labels,average=None, labels=['NEG','NEU','POS']) # TP/(TP+FN) i.e the ability of the estimator to predict all the tweets of a given class

print(recall)

## 5.4 F1 Score

In [ ]:
f1score = f1_score(label_test, predicted_labels,average=None, labels=['NEG','NEU','POS']) # 2*(precision*recall)/(precision+recall)

print(f1score)

# 7. Push To Hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import push_to_hub_keras

push_to_hub_keras(model, 'FedeBerto/Griffith-Sentiment')

In [ ]:
from huggingface_hub import from_pretrained_keras

model = from_pretrained_keras('FedeBerto/Griffith-Sentiment')